In [10]:
import pandas as pd
from itertools import combinations
from statsmodels.stats.multitest import multipletests
from scipy import stats

In [2]:

experiment_m = pd.read_csv("../data/ab_testing/experiment_m.csv")
groups = sorted(experiment_m["group"].dropna().unique())

In [ ]:
for i in combinations(groups, 2):
    print(i)

('Test1', 'Test2')
('Test1', 'Test3')
('Test1', 'Test4')
('Test1', 'Test5')
('Test1', 'Test6')
('Test1', 'Test7')
('Test1', 'Test8')
('Test1', 'Test9')
('Test1', 'control')
('Test2', 'Test3')
('Test2', 'Test4')
('Test2', 'Test5')
('Test2', 'Test6')
('Test2', 'Test7')
('Test2', 'Test8')
('Test2', 'Test9')
('Test2', 'control')
('Test3', 'Test4')
('Test3', 'Test5')
('Test3', 'Test6')
('Test3', 'Test7')
('Test3', 'Test8')
('Test3', 'Test9')
('Test3', 'control')
('Test4', 'Test5')
('Test4', 'Test6')
('Test4', 'Test7')
('Test4', 'Test8')
('Test4', 'Test9')
('Test4', 'control')
('Test5', 'Test6')
('Test5', 'Test7')
('Test5', 'Test8')
('Test5', 'Test9')
('Test5', 'control')
('Test6', 'Test7')
('Test6', 'Test8')
('Test6', 'Test9')
('Test6', 'control')
('Test7', 'Test8')
('Test7', 'Test9')
('Test7', 'control')
('Test8', 'Test9')
('Test8', 'control')
('Test9', 'control')


Pairwise T-test

In [11]:

def pairwise_ttests(data, value_col, group_col):
    groups = sorted(data[group_col].dropna().unique())
    rows = []

    for g1, g2 in combinations(groups, 2):
        x = data.loc[data[group_col] == g1, value_col].dropna()
        y = data.loc[data[group_col] == g2, value_col].dropna()

        t_stat, raw_p = stats.ttest_ind(
            x, y,
            equal_var=False,
            nan_policy="omit"
        )

        rows.append({
            "group_1": g1,
            "group_2": g2,
            "t_statistic": t_stat,
            "raw_p_value": raw_p
        })

    return pd.DataFrame(rows)

In [12]:
pairwise_results = pairwise_ttests(
    experiment_m,
    value_col="viewing_time",
    group_col="group"
)

pairwise_results.head()

,group_1,group_2,t_statistic,raw_p_value
0,Test1,Test2,-2.845693,0.004686
1,Test1,Test3,-1.917291,0.055942
2,Test1,Test4,-2.799258,0.005391
3,Test1,Test5,0.279632,0.779907
4,Test1,Test6,-3.281141,0.001125


In [4]:
group_counts = (
    experiment_m.groupby("group", as_index=False)
    .size()
    .rename(columns={"size": "n"})
)

group_counts

,group,n
0,Test1,200
1,Test2,200
2,Test3,200
3,Test4,200
4,Test5,200
5,Test6,200
6,Test7,200
7,Test8,200
8,Test9,200
9,control,200


In [5]:
experiment_m.head()

,viewing_time,group
0,15.656666,control
1,70.875219,control
2,31.530482,control
3,66.033616,control
4,34.834495,control


In [13]:
pairwise_results = pairwise_ttests(
    experiment_m,
    value_col="viewing_time",
    group_col="group"
)

pairwise_results.head()

,group_1,group_2,t_statistic,raw_p_value
0,Test1,Test2,-2.845693,0.004686
1,Test1,Test3,-1.917291,0.055942
2,Test1,Test4,-2.799258,0.005391
3,Test1,Test5,0.279632,0.779907
4,Test1,Test6,-3.281141,0.001125


In [17]:
import this

The Zen of Python, by Tim Peters

Beautiful is better than ugly.
Explicit is better than implicit.
Simple is better than complex.
Complex is better than complicated.
Flat is better than nested.
Sparse is better than dense.
Readability counts.
Special cases aren't special enough to break the rules.
Although practicality beats purity.
Errors should never pass silently.
Unless explicitly silenced.
In the face of ambiguity, refuse the temptation to guess.
There should be one-- and preferably only one --obvious way to do it.
Although that way may not be obvious at first unless you're Dutch.
Now is better than never.
Although never is often better than *right* now.
If the implementation is hard to explain, it's a bad idea.
If the implementation is easy to explain, it may be a good idea.
Namespaces are one honking great idea -- let's do more of those!


In [29]:
significant_values= (pairwise_results[pairwise_results['raw_p_value']<=0.05]
 .sort_values(by = ['raw_p_value']))

significant_values.shape

(24, 4)

In [32]:
multipletests(pairwise_results["raw_p_value"], method="bonferroni")

(array([False, False, False, False, False, False, False, False, False,
        False, False, False, False, False,  True, False,  True, False,
        False, False, False,  True, False,  True, False, False, False,
         True, False,  True,  True, False, False, False, False, False,
         True, False,  True, False, False, False, False, False, False]),
 array([2.10853122e-01, 1.00000000e+00, 2.42572919e-01, 1.00000000e+00,
        5.06274717e-02, 1.00000000e+00, 1.00000000e+00, 1.00000000e+00,
        1.00000000e+00, 1.00000000e+00, 1.00000000e+00, 1.36293201e-01,
        1.00000000e+00, 5.62993443e-01, 4.09756793e-04, 1.00000000e+00,
        1.89623501e-03, 1.00000000e+00, 1.00000000e+00, 1.00000000e+00,
        1.00000000e+00, 9.00505741e-03, 1.00000000e+00, 3.55830445e-02,
        1.56239210e-01, 1.00000000e+00, 6.62759228e-01, 3.92320044e-04,
        1.00000000e+00, 1.95677492e-03, 3.30911183e-02, 1.00000000e+00,
        1.00000000e+00, 1.00000000e+00, 1.00000000e+00, 1.98841974e

In [ ]:
multipletests(pairwise_results["raw_p_value"], method="bonferroni")[1]


array([2.10853122e-01, 1.00000000e+00, 2.42572919e-01, 1.00000000e+00,
       5.06274717e-02, 1.00000000e+00, 1.00000000e+00, 1.00000000e+00,
       1.00000000e+00, 1.00000000e+00, 1.00000000e+00, 1.36293201e-01,
       1.00000000e+00, 5.62993443e-01, 4.09756793e-04, 1.00000000e+00,
       1.89623501e-03, 1.00000000e+00, 1.00000000e+00, 1.00000000e+00,
       1.00000000e+00, 9.00505741e-03, 1.00000000e+00, 3.55830445e-02,
       1.56239210e-01, 1.00000000e+00, 6.62759228e-01, 3.92320044e-04,
       1.00000000e+00, 1.95677492e-03, 3.30911183e-02, 1.00000000e+00,
       1.00000000e+00, 1.00000000e+00, 1.00000000e+00, 1.98841974e-01,
       1.46007714e-05, 1.00000000e+00, 1.50256519e-04, 6.40851611e-01,
       1.00000000e+00, 1.00000000e+00, 3.90300824e-01, 1.00000000e+00,
       9.27407752e-01])

Editing with `Bonferoni` method, which assumdes:

$$p_value <= frac{\alpha}{m}$$

In [33]:
pairwise_results["p_value_adjusted"] = multipletests(pairwise_results["raw_p_value"], 
                                                     method="bonferroni")[1]

In [36]:
pairwise_results.head(11)

,group_1,group_2,t_statistic,raw_p_value,p_value_adjusted
0,Test1,Test2,-2.845693,0.004686,0.210853
1,Test1,Test3,-1.917291,0.055942,1.000000
2,Test1,Test4,-2.799258,0.005391,0.242573
3,Test1,Test5,0.279632,0.779907,1.000000
4,Test1,Test6,-3.281141,0.001125,0.050627
5,Test1,Test7,-0.382770,0.702095,1.000000
6,Test1,Test8,2.123086,0.034372,1.000000
7,Test1,Test9,-0.829841,0.407162,1.000000
8,Test1,control,1.774162,0.076830,1.000000
9,Test2,Test3,1.025306,0.305856,1.000000


In [ ]:
significant_values_adj= (pairwise_results[pairwise_results['p_value_adjusted']<=0.05]
 .sort_values(by = ['p_value_adjusted']))



(24, 4)

In [39]:
print(significant_values.shape)
print(significant_values_adj.shape)

(24, 4)
(9, 5)


In [41]:
import pingouin as pg
experiment_m.head()

,viewing_time,group
0,15.656666,control
1,70.875219,control
2,31.530482,control
3,66.033616,control
4,34.834495,control


In [44]:
ptt = pg.pairwise_tests(data= experiment_m, 
                        dv = 'viewing_time', 
                        between = 'group', 
                        padjust ='bonf')

In [45]:
ptt.head()

,Contrast,A,B,Paired,Parametric,T,dof,alternative,p_unc,p_corr,p_adjust,BF10,hedges
0,group,Test1,Test2,False,True,-2.845693,398.0,two-sided,0.004661,0.209728,bonf,5.357,-0.284033
1,group,Test1,Test3,False,True,-1.917291,398.0,two-sided,0.055917,1.000000,bonf,0.649,-0.191368
2,group,Test1,Test4,False,True,-2.799258,398.0,two-sided,0.005371,0.241709,bonf,4.73,-0.279398
3,group,Test1,Test5,False,True,0.279632,398.0,two-sided,0.779905,1.000000,bonf,0.115,0.027910
4,group,Test1,Test6,False,True,-3.281141,398.0,two-sided,0.001125,0.050627,bonf,18.964,-0.327495


In [46]:
pairwise_results

,group_1,group_2,t_statistic,raw_p_value,p_value_adjusted
0,Test1,Test2,-2.845693,4.685625e-03,0.210853
1,Test1,Test3,-1.917291,5.594173e-02,1.000000
2,Test1,Test4,-2.799258,5.390509e-03,0.242573
3,Test1,Test5,0.279632,7.799066e-01,1.000000
4,Test1,Test6,-3.281141,1.125055e-03,0.050627
5,Test1,Test7,-0.382770,7.020949e-01,1.000000
6,Test1,Test8,2.123086,3.437227e-02,1.000000
7,Test1,Test9,-0.829841,4.071620e-01,1.000000
8,Test1,control,1.774162,7.682968e-02,1.000000
9,Test2,Test3,1.025306,3.058557e-01,1.000000
